# 10a. Validation Data Preparation

**Role of this notebook.** Build the semi-synthetic validation dataset used to compare recommender systems under a known true utility surface. The notebook does not estimate model parameters. It simulates random-exploration interactions and creates a separate holdout candidate set for each user.

**Steps.**

1. Load feature schemas, calibrated utility objects, structural parameters, and output paths.
2. Load the calibrated true utility table and static user/video features.
3. Define the structural threshold solver and the simulated watch-ratio outcome generator.
4. Simulate 2,000 exploration interactions per user with online history updates.
5. Sample holdout candidates and save the feature manifest, config, and diagnostics.

The output tables are the common inputs for the validation recommenders:

| Output | Purpose |
|---|---|
| `validation_exploration_interactions.parquet` | simulated random-exploration rows with labels, history, beliefs, and scalable features |
| `validation_holdout_candidates.parquet` | candidate videos to rank for each user during recommender evaluation |
| `validation_feature_manifest.json` | feature groups used by downstream MLP models |
| `validation_simulation_config.json` | simulation constants and input/output paths |
| `validation_simulation_diagnostics.json` | sanity checks on row counts, outcomes, and true utility |


In [1]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.special import ndtr

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
OUTPUT_DIR = PROJECT_ROOT / "python_code_new" / "outputs"
PROCESSED_DATA_PATH = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed" / "processed_data.parquet"
GNN_DATA_PATH = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed" / "gnn_data.pt"
TRUE_UTILITY_PATH = OUTPUT_DIR / "semi_synth_true_utility.parquet"
UTILITY_PARAM_PATH = OUTPUT_DIR / "semi_synth_utility_params.npz"
STRUCTURAL_PARAM_PATH = OUTPUT_DIR / "structural_estimates_theta_phi.npz"

SIM_OUTPUT_DIR = OUTPUT_DIR / "validation_simulation"
SIM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPLORATION_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_exploration_interactions.parquet"
HOLDOUT_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_holdout_candidates.parquet"
CONFIG_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_simulation_config.json"
MANIFEST_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_feature_manifest.json"
DIAGNOSTICS_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_simulation_diagnostics.json"

RANDOM_SEED = 20260623
NUM_EXPLORATION_VIDEOS_PER_USER = 2000
NUM_HOLDOUT_VIDEOS_PER_USER = 1000
VIDEOS_PER_SESSION = 10
NUM_SESSIONS = NUM_EXPLORATION_VIDEOS_PER_USER // VIDEOS_PER_SESSION
PRIOR_STRENGTH = 5.0
KAPPA = 0.25
SIM_BURST_ID = 1
SIM_SESSION_GAP_HOURS = 1.25
EMA_ALPHA = 0.1
MIN_VAR = 1e-4
THRESHOLD_COST_FLOOR = 1e-8
WRITE_CHUNK_USERS = 50

# Keep simulated measurement/history values inside the empirical support used to estimate phi.
# These are from processed_data: watch_ratio p99.5 and hist_ema_watchratio p95.
LOG_WR_CLIP_LO = -8.0
LOG_WR_CLIP_HI = 1.935523
HIST_EMA_WATCHRATIO_CAP = 1.682821

COMPLETE_WATCH_RATIO = 1.0
LONG_WATCH_RATIO = 1.5
LONG_PLAY_MS = 12_000
REWATCH_WATCH_RATIO = 2.0
NEGATIVE_PLAY_MS = 2_000

if NUM_EXPLORATION_VIDEOS_PER_USER % VIDEOS_PER_SESSION != 0:
    raise ValueError("NUM_EXPLORATION_VIDEOS_PER_USER must be divisible by VIDEOS_PER_SESSION")

for required_path in [PROCESSED_DATA_PATH, GNN_DATA_PATH, TRUE_UTILITY_PATH, UTILITY_PARAM_PATH, STRUCTURAL_PARAM_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print("output dir:", SIM_OUTPUT_DIR)

output dir: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation


## 1. Setup and Inputs

This section fixes all paths and random seeds, then loads the feature schema and structural objects needed by the simulator. The key inputs are:

| Input | Use |
|---|---|
| processed KuaiRec table | static user/video attributes and original feature metadata |
| calibrated utility files | true user-video utility `u_star`, utility mean `mu_hat`, and variance `sigma_hat` |
| structural estimates | search-cost and watch-ratio measurement parameters used to simulate behavior |
| GNN feature metadata | continuous, binary, categorical, and belief feature definitions |

The validation simulator treats calibrated `u_star` as the ground truth. Downstream recommenders may use simulated outcomes and serving-safe features, but they should not use `u_star` for fitting.


In [2]:
import torch

gnn_data = torch.load(GNN_DATA_PATH, map_location="cpu", weights_only=False)
feature_metadata = gnn_data["feature_metadata"]
continuous_cols = list(feature_metadata["continuous_cols"])
binary_cols = list(feature_metadata["binary_cols"])
categorical_cols = list(feature_metadata["categorical_cols"])
mlp_feature_columns = continuous_cols + binary_cols + categorical_cols

continuous_missing = dict(feature_metadata.get("continuous_missing", {}))  # missing counts, not imputation values
binary_missing = dict(feature_metadata.get("binary_missing", {}))  # missing counts, not imputation values
continuous_default = {
    col: float(feature_metadata.get("continuous_scaler", {}).get(col, {}).get("mean", 0.0))
    for col in continuous_cols
}
binary_default = {col: 0 for col in binary_cols}

utility_npz = np.load(UTILITY_PARAM_PATH)
user_ids = utility_npz["user_ids"].astype(np.int64)
cat_ids = utility_npz["cat_ids"].astype(np.int64)
mu_matrix = utility_npz["mu_matrix"].astype(np.float64)
sigma_matrix = utility_npz["sigmas_by_user"].astype(np.float64)

struct_npz = np.load(STRUCTURAL_PARAM_PATH)
struct_user_ids = struct_npz["user_ids"].astype(np.int64)
if not np.array_equal(user_ids, struct_user_ids):
    raise ValueError("Notebook 08 user_ids and notebook 06 structural user_ids are not aligned")

cost_by_user = np.maximum(struct_npz["theta_c"].astype(np.float64), THRESHOLD_COST_FLOOR)
cat_id_to_idx = {int(cat_id): idx for idx, cat_id in enumerate(cat_ids)}

phi = {
    "alpha_mu0": struct_npz["phi_alpha_mu0"].astype(np.float64),
    "alpha_mu1": struct_npz["phi_alpha_mu1"].astype(np.float64),
    "beta_mu0": struct_npz["phi_beta_mu0"].astype(np.float64),
    "beta_mu1": struct_npz["phi_beta_mu1"].astype(np.float64),
    "alpha_logvar0": struct_npz["phi_alpha_logvar0"].astype(np.float64),
    "alpha_logvar1": struct_npz["phi_alpha_logvar1"].astype(np.float64),
    "beta_logvar0": struct_npz["phi_beta_logvar0"].astype(np.float64),
    "beta_logvar1": struct_npz["phi_beta_logvar1"].astype(np.float64),
}

mean_feature_names = [
    "log1p_i_video_duration",
    "sess_rank_over_sess_len",
    "sess_rank_over_sess_len_sq",
    "ctx_hour_sin",
    "ctx_hour_cos",
    "ctx_is_weekend",
    "u_is_lowactive_period",
    "hist_has_author_history",
    "hist_ema_watchratio",
]
logvar_feature_names = [
    "log1p_i_video_duration",
    "sess_rank_over_sess_len",
    "ctx_hour_sin",
    "ctx_hour_cos",
    "ctx_is_weekend",
]

print("users:", len(user_ids))
print("level-1 categories:", len(cat_ids))
print("MLP features:", len(mlp_feature_columns), "=", len(continuous_cols), "continuous +", len(binary_cols), "binary +", len(categorical_cols), "categorical")
print("measurement mean features:", mean_feature_names)
print("measurement logvar features:", logvar_feature_names)

users: 1777
level-1 categories: 39
MLP features: 64 = 23 continuous + 6 binary + 35 categorical
measurement mean features: ['log1p_i_video_duration', 'sess_rank_over_sess_len', 'sess_rank_over_sess_len_sq', 'ctx_hour_sin', 'ctx_hour_cos', 'ctx_is_weekend', 'u_is_lowactive_period', 'hist_has_author_history', 'hist_ema_watchratio']
measurement logvar features: ['log1p_i_video_duration', 'sess_rank_over_sess_len', 'ctx_hour_sin', 'ctx_hour_cos', 'ctx_is_weekend']


## 2. True Utility and Static Features

This section constructs one reusable row per observed user-video pair in the calibrated utility surface. The true utility table provides:

| Column | Meaning |
|---|---|
| `mu_hat` | calibrated conditional mean utility for the user-video category |
| `sigma_hat` | calibrated conditional utility standard deviation |
| `u_star` | realized true utility used as the validation target |
| category ids | level-1, level-2, and level-3 video categories used by beliefs and utility moments |

Static user and video covariates are then joined from the processed interaction data. History covariates are not static; they are recomputed during the simulation loop so each row only sees information available before that simulated interaction.


In [3]:
true_cols = [
    "user_id", "video_id", "level1_category", "level2_category", "level3_category",
    "mu_hat", "sigma_hat", "u_star",
]
true_df = pd.read_parquet(TRUE_UTILITY_PATH, columns=true_cols)

counts_by_user = true_df.groupby("user_id", sort=False).size().to_numpy()
if counts_by_user.min() != counts_by_user.max():
    raise ValueError("True utility surface is not a balanced user-video catalog")

N_users = len(user_ids)
N_videos = int(counts_by_user[0])
if len(true_df) != N_users * N_videos:
    raise ValueError("True utility surface row count does not equal N_users * N_videos")

first_user = user_ids[0]
video_catalog = (
    true_df.loc[true_df["user_id"].eq(first_user), ["video_id", "level1_category", "level2_category", "level3_category"]]
    .reset_index(drop=True)
    .copy()
)
video_ids = video_catalog["video_id"].to_numpy(dtype=np.int64)
video_id_to_col = {int(v): idx for idx, v in enumerate(video_ids)}

# Validate that each user block has the same video order. This lets us reshape utility arrays safely.
for check_user in user_ids[::max(1, len(user_ids) // 10)]:
    vids = true_df.loc[true_df["user_id"].eq(check_user), "video_id"].to_numpy(dtype=np.int64)
    if not np.array_equal(vids, video_ids):
        raise ValueError(f"Video catalog order differs for user_id={check_user}")

u_star_matrix = true_df["u_star"].to_numpy(dtype=np.float32).reshape(N_users, N_videos)
mu_obs_matrix = true_df["mu_hat"].to_numpy(dtype=np.float32).reshape(N_users, N_videos)
sigma_obs_matrix = true_df["sigma_hat"].to_numpy(dtype=np.float32).reshape(N_users, N_videos)

processed_cols = sorted(set(["user_id", "video_id"] + mlp_feature_columns + [
    "i_author_id", "i_cat_level1_id", "i_cat_level2_id", "i_cat_level3_id",
    "i_video_duration", "i_age_since_upload_days", "i_aspect_ratio",
]))
processed_df = pd.read_parquet(PROCESSED_DATA_PATH, columns=processed_cols)

user_static_cols = [c for c in mlp_feature_columns if c.startswith("u_")]
video_static_cols = [
    c for c in mlp_feature_columns
    if c.startswith("i_") or c in ["i_author_id", "i_video_type", "i_upload_type", "i_visible_status", "i_music_id", "i_video_tag_id", "i_video_tag_name"]
]
video_static_cols = sorted(set(video_static_cols + ["i_author_id", "i_cat_level1_id", "i_cat_level2_id", "i_cat_level3_id", "i_video_duration", "i_age_since_upload_days", "i_aspect_ratio"]))

user_static_df = processed_df[["user_id"] + user_static_cols].drop_duplicates("user_id").set_index("user_id").reindex(user_ids).reset_index()
video_static_df = processed_df[["video_id"] + video_static_cols].drop_duplicates("video_id").set_index("video_id").reindex(video_ids).reset_index()

# Use notebook 08's category IDs as the authoritative category hierarchy.
video_static_df = video_static_df.drop(columns=[c for c in ["i_cat_level1_id", "i_cat_level2_id", "i_cat_level3_id"] if c in video_static_df.columns])
video_static_df = video_static_df.merge(
    video_catalog.rename(columns={
        "level1_category": "i_cat_level1_id",
        "level2_category": "i_cat_level2_id",
        "level3_category": "i_cat_level3_id",
    }),
    on="video_id",
    how="left",
    validate="one_to_one",
)

# Fill missing serving-safe feature values using notebook 02's recorded defaults where available.
for col in continuous_cols:
    if col in user_static_df.columns:
        user_static_df[col] = pd.to_numeric(user_static_df[col], errors="coerce").fillna(float(continuous_default.get(col, 0.0)))
    if col in video_static_df.columns:
        video_static_df[col] = pd.to_numeric(video_static_df[col], errors="coerce").fillna(float(continuous_default.get(col, 0.0)))
for col in binary_cols:
    if col in user_static_df.columns:
        user_static_df[col] = pd.to_numeric(user_static_df[col], errors="coerce").fillna(int(binary_default.get(col, 0))).astype("int8")
    if col in video_static_df.columns:
        video_static_df[col] = pd.to_numeric(video_static_df[col], errors="coerce").fillna(int(binary_default.get(col, 0))).astype("int8")

# Store row dictionaries for fast simulation lookup.
user_static_by_id = user_static_df.set_index("user_id").to_dict(orient="index")
video_static_by_col = video_static_df.reset_index(drop=True)
video_static_records = video_static_by_col.to_dict(orient="records")

print("true utility matrix:", u_star_matrix.shape)
print("video catalog:", video_catalog.shape)
print("user static table:", user_static_df.shape)
print("video static table:", video_static_df.shape)

true utility matrix: (1777, 10728)
video catalog: (10728, 4)
user static table: (1777, 35)
video static table: (10728, 14)


## 3. Structural Simulation Functions

For each user-session, the simulator solves a search threshold from the belief-weighted utility distribution. Given category belief vector `B_it`, category means `mu_ik`, standard deviations `sigma_ik`, and search cost `c_i`, the threshold `tau_it` solves:

$$
\sum_k B_{itk}\,E[(U_{ik}-\tau_{it})_+] = c_i.
$$

For a normal component, the expected excess utility is:

$$
E[(U-\tau)_+] = \sigma \phi(a) + (\mu-\tau)[1-\Phi(a)],
\quad
 a = \frac{\tau-\mu}{\sigma}.
$$

After the threshold is solved, the latent high-utility state is:

$$
z_{itj}=1\{u_{ij} > \tau_{it}\}.
$$

The watch-ratio outcome is drawn from the calibrated measurement equation:

$$
\log(WR_{itj}) = m_{z,itj} + \varepsilon_{itj},
\quad
\varepsilon_{itj}\sim N(0, v_{z,itj}).
$$

The simulator then maps watch ratio into the four behavior heads used by the baseline recommender: complete, long-watch, rewatch, and negative/short-watch.


In [4]:
def normal_pdf(x):
    x = np.asarray(x, dtype=np.float64)
    return np.exp(-0.5 * x * x) / np.sqrt(2.0 * np.pi)


def expected_marginal_gain(tau, mus, sigmas, belief):
    sigmas = np.maximum(np.asarray(sigmas, dtype=np.float64), 1e-6)
    mus = np.asarray(mus, dtype=np.float64)
    belief = np.asarray(belief, dtype=np.float64)
    a = (tau - mus) / sigmas
    tail = np.clip(1.0 - ndtr(a), 1e-300, 1.0)
    gain_by_cat = (mus - tau) * tail + sigmas * normal_pdf(a)
    return float(np.sum(belief * gain_by_cat))


def solve_tau_single(mus, sigmas, belief, cost, n_iter=80):
    cost = max(float(cost), THRESHOLD_COST_FLOOR)
    mus = np.asarray(mus, dtype=np.float64)
    sigmas = np.maximum(np.asarray(sigmas, dtype=np.float64), 1e-6)
    lo = float(np.min(mus - 12.0 * sigmas))
    hi = float(np.max(mus + 12.0 * sigmas))

    for _ in range(20):
        if expected_marginal_gain(lo, mus, sigmas, belief) >= cost:
            break
        lo -= float(np.max(sigmas) * 4.0 + 1.0)
    for _ in range(20):
        if expected_marginal_gain(hi, mus, sigmas, belief) <= cost:
            break
        hi += float(np.max(sigmas) * 4.0 + 1.0)

    for _ in range(n_iter):
        mid = 0.5 * (lo + hi)
        if expected_marginal_gain(mid, mus, sigmas, belief) > cost:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def expected_marginal_gain_matrix(tau, mus, sigmas, beliefs):
    tau = np.asarray(tau, dtype=np.float64).reshape(-1, 1)
    mus = np.asarray(mus, dtype=np.float64).reshape(1, -1)
    sigmas = np.maximum(np.asarray(sigmas, dtype=np.float64).reshape(1, -1), 1e-6)
    beliefs = np.asarray(beliefs, dtype=np.float64)
    a = (tau - mus) / sigmas
    tail = np.clip(1.0 - ndtr(a), 1e-300, 1.0)
    gain_by_cat = (mus - tau) * tail + sigmas * normal_pdf(a)
    return np.sum(beliefs * gain_by_cat, axis=1)


def solve_tau_vectorized_user(mus, sigmas, beliefs, cost, n_iter=50):
    beliefs = np.asarray(beliefs, dtype=np.float64)
    n_sessions = beliefs.shape[0]
    cost_vec = np.full(n_sessions, max(float(cost), THRESHOLD_COST_FLOOR), dtype=np.float64)
    mus = np.asarray(mus, dtype=np.float64)
    sigmas = np.maximum(np.asarray(sigmas, dtype=np.float64), 1e-6)
    lo = np.full(n_sessions, float(np.min(mus - 12.0 * sigmas)), dtype=np.float64)
    hi = np.full(n_sessions, float(np.max(mus + 12.0 * sigmas)), dtype=np.float64)
    step = float(np.max(sigmas) * 4.0 + 1.0)

    for _ in range(20):
        bad_lo = expected_marginal_gain_matrix(lo, mus, sigmas, beliefs) < cost_vec
        if not bad_lo.any():
            break
        lo[bad_lo] -= step
    for _ in range(20):
        bad_hi = expected_marginal_gain_matrix(hi, mus, sigmas, beliefs) > cost_vec
        if not bad_hi.any():
            break
        hi[bad_hi] += step

    for _ in range(n_iter):
        mid = 0.5 * (lo + hi)
        should_raise = expected_marginal_gain_matrix(mid, mus, sigmas, beliefs) > cost_vec
        lo = np.where(should_raise, mid, lo)
        hi = np.where(should_raise, hi, mid)
    return 0.5 * (lo + hi)


def stable_log_variance_from_eta(eta, min_var=MIN_VAR):
    return np.logaddexp(np.asarray(eta, dtype=np.float64), np.log(float(min_var)))


def build_watch_feature_arrays(frame):
    duration = pd.to_numeric(frame["i_video_duration"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
    log_duration = np.log1p(np.maximum(duration, 0.0))

    sess_rank = pd.to_numeric(frame["sess_rank"], errors="coerce").fillna(1.0).to_numpy(dtype=np.float64)
    sess_len = pd.to_numeric(frame["sess_len"], errors="coerce").fillna(VIDEOS_PER_SESSION).to_numpy(dtype=np.float64)
    rel_rank = sess_rank / np.maximum(sess_len, 1.0)
    rel_rank_sq = rel_rank * rel_rank

    hour_sin = pd.to_numeric(frame["ctx_hour_sin"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
    hour_cos = pd.to_numeric(frame["ctx_hour_cos"], errors="coerce").fillna(1.0).to_numpy(dtype=np.float64)
    weekend = pd.to_numeric(frame["ctx_is_weekend"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
    low_activity = pd.to_numeric(frame["u_is_lowactive_period"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
    has_author_history = pd.to_numeric(frame["hist_has_author_history"], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
    ema_watchratio = pd.to_numeric(frame["hist_ema_watchratio"], errors="coerce").fillna(float(continuous_default.get("hist_ema_watchratio", 0.0))).to_numpy(dtype=np.float64)

    x_mean = np.vstack([
        log_duration, rel_rank, rel_rank_sq, hour_sin, hour_cos, weekend,
        low_activity, has_author_history, ema_watchratio,
    ]).T
    x_logvar = np.vstack([log_duration, rel_rank, hour_sin, hour_cos, weekend]).T
    return x_mean, x_logvar


def predict_watch_moments(frame):
    x_mean, x_logvar = build_watch_feature_arrays(frame)
    cat_idx = frame["cat_idx_int"].to_numpy(dtype=np.int64)
    m0 = phi["alpha_mu0"][cat_idx] + x_mean @ phi["beta_mu0"]
    m1 = phi["alpha_mu1"][cat_idx] + x_mean @ phi["beta_mu1"]
    eta0 = phi["alpha_logvar0"][cat_idx] + x_logvar @ phi["beta_logvar0"]
    eta1 = phi["alpha_logvar1"][cat_idx] + x_logvar @ phi["beta_logvar1"]
    logv0 = stable_log_variance_from_eta(eta0)
    logv1 = stable_log_variance_from_eta(eta1)
    return m0, logv0, m1, logv1


def predict_watch_moments_scalar(row, cat_idx):
    duration = max(float(row.get("i_video_duration", 0.0) or 0.0), 0.0)
    log_duration = math.log1p(duration)
    sess_rank = float(row.get("sess_rank", 1.0) or 1.0)
    sess_len = max(float(row.get("sess_len", VIDEOS_PER_SESSION) or VIDEOS_PER_SESSION), 1.0)
    rel_rank = sess_rank / sess_len
    x_mean = np.array([
        log_duration,
        rel_rank,
        rel_rank * rel_rank,
        float(row.get("ctx_hour_sin", 0.0) or 0.0),
        float(row.get("ctx_hour_cos", 1.0) or 1.0),
        float(row.get("ctx_is_weekend", 0.0) or 0.0),
        float(row.get("u_is_lowactive_period", 0.0) or 0.0),
        float(row.get("hist_has_author_history", 0.0) or 0.0),
        float(row.get("hist_ema_watchratio", continuous_default.get("hist_ema_watchratio", 0.0)) or 0.0),
    ], dtype=np.float64)
    x_logvar = np.array([
        log_duration,
        rel_rank,
        float(row.get("ctx_hour_sin", 0.0) or 0.0),
        float(row.get("ctx_hour_cos", 1.0) or 1.0),
        float(row.get("ctx_is_weekend", 0.0) or 0.0),
    ], dtype=np.float64)
    m0 = float(phi["alpha_mu0"][cat_idx] + x_mean @ phi["beta_mu0"])
    m1 = float(phi["alpha_mu1"][cat_idx] + x_mean @ phi["beta_mu1"])
    eta0 = float(phi["alpha_logvar0"][cat_idx] + x_logvar @ phi["beta_logvar0"])
    eta1 = float(phi["alpha_logvar1"][cat_idx] + x_logvar @ phi["beta_logvar1"])
    logv0 = float(stable_log_variance_from_eta(eta0))
    logv1 = float(stable_log_variance_from_eta(eta1))
    return m0, logv0, m1, logv1


def entropy_from_counts(counts):
    counts = np.asarray(list(counts), dtype=np.float64)
    total = counts.sum()
    if total <= 0:
        return 0.0
    p = counts[counts > 0] / total
    return float(-(p * np.log(p)).sum())


def sigmoid_stable(x):
    x = np.asarray(x, dtype=np.float64)
    return np.where(x >= 0, 1.0 / (1.0 + np.exp(-x)), np.exp(x) / (1.0 + np.exp(x)))


def make_context_for_session(session_idx):
    base = pd.Timestamp("2020-09-06 08:00:00")
    t = base + pd.Timedelta(hours=float(session_idx) * SIM_SESSION_GAP_HOURS)
    hour = t.hour + t.minute / 60.0 + t.second / 3600.0
    angle = 2.0 * np.pi * hour / 24.0
    return {
        "time": t,
        "date": int(t.strftime("%Y%m%d")),
        "timestamp": float(t.timestamp()),
        "ctx_hour_sin": float(np.sin(angle)),
        "ctx_hour_cos": float(np.cos(angle)),
        "ctx_is_weekend": int(t.dayofweek >= 5),
    }


def append_parquet(df, path, writer_holder):
    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer_holder.get("writer") is None:
        writer_holder["writer"] = pq.ParquetWriter(path, table.schema, compression="zstd")
    writer_holder["writer"].write_table(table)


def close_writer(writer_holder):
    writer = writer_holder.get("writer")
    if writer is not None:
        writer.close()
        writer_holder["writer"] = None

## 4. Simulate Exploration and Holdout Rows

This section creates the validation data user by user.

For each user:

1. Draw `2,000` random-exploration videos and split them into sessions of 10 videos.
2. Record pre-interaction features: current session context, history EMAs, category history, author history, and belief vector.
3. Solve the session threshold and draw the simulated watch-ratio outcome from the structural measurement model.
4. Convert the simulated watch ratio into the four behavior labels.
5. Update the user's simulated history state before moving to the next row.
6. Draw a separate holdout candidate set that is not used to fit any recommender.

The important timing rule is that each row's history features are measured before that row's simulated outcome is drawn.


In [5]:
rng = np.random.default_rng(RANDOM_SEED)

# Clear stale outputs before writing a fresh simulation.
for output_path in [EXPLORATION_OUTPUT_PATH, HOLDOUT_OUTPUT_PATH]:
    if output_path.exists():
        output_path.unlink()

level1_col = video_catalog["level1_category"].to_numpy(dtype=np.int64)
level2_col = video_catalog["level2_category"].to_numpy(dtype=np.int64)
level3_col = video_catalog["level3_category"].to_numpy(dtype=np.int64)
cat_idx_col = np.array([cat_id_to_idx[int(cat)] for cat in level1_col], dtype=np.int64)
global_level1_counts = np.bincount(cat_idx_col, minlength=len(cat_ids)).astype(np.float64)
global_level1_dist = global_level1_counts / global_level1_counts.sum()

exploration_writer = {"writer": None}
holdout_writer = {"writer": None}

all_diag_rows = []
session_diag_rows = []
category_exploration_counts = np.zeros(len(cat_ids), dtype=np.int64)
category_holdout_counts = np.zeros(len(cat_ids), dtype=np.int64)

for user_start in range(0, N_users, WRITE_CHUNK_USERS):
    user_stop = min(user_start + WRITE_CHUNK_USERS, N_users)
    exploration_chunks = []
    holdout_chunks = []

    for user_idx in range(user_start, user_stop):
        user_id = int(user_ids[user_idx])
        sampled_cols = rng.choice(N_videos, size=NUM_EXPLORATION_VIDEOS_PER_USER + NUM_HOLDOUT_VIDEOS_PER_USER, replace=False)
        exploration_cols = sampled_cols[:NUM_EXPLORATION_VIDEOS_PER_USER]
        holdout_cols = sampled_cols[NUM_EXPLORATION_VIDEOS_PER_USER:]

        alpha_counts = PRIOR_STRENGTH * global_level1_dist.copy()
        ema_values = {
            "hist_ema_y_complete": float(continuous_default.get("hist_ema_y_complete", 0.0)),
            "hist_ema_y_long": float(continuous_default.get("hist_ema_y_long", 0.0)),
            "hist_ema_y_rewatch": float(continuous_default.get("hist_ema_y_rewatch", 0.0)),
            "hist_ema_y_neg": float(continuous_default.get("hist_ema_y_neg", 0.0)),
            "hist_ema_watchratio": float(continuous_default.get("hist_ema_watchratio", 0.0)),
        }
        cat_complete_ema = {int(cat): float(continuous_default.get("hist_cat_ema_complete", 0.0)) for cat in cat_ids}
        level2_seen_counts = {}
        author_last_complete = {}
        author_last_time = {}
        prev_session_len = float(continuous_default.get("hist_prev_sess_len", 0.0))
        prev_session_end_time = None

        session_cols_matrix = exploration_cols.reshape(NUM_SESSIONS, VIDEOS_PER_SESSION)
        alpha_before_sessions = np.empty((NUM_SESSIONS, len(cat_ids)), dtype=np.float64)
        alpha_after_sessions = np.empty((NUM_SESSIONS, len(cat_ids)), dtype=np.float64)
        belief_before_sessions = np.empty((NUM_SESSIONS, len(cat_ids)), dtype=np.float64)
        belief_after_sessions = np.empty((NUM_SESSIONS, len(cat_ids)), dtype=np.float64)
        alpha_work = alpha_counts.copy()
        for s in range(NUM_SESSIONS):
            alpha_before_sessions[s] = alpha_work
            belief_before_sessions[s] = alpha_work / alpha_work.sum()
            counts_s = np.bincount(cat_idx_col[session_cols_matrix[s]], minlength=len(cat_ids))
            alpha_work = alpha_work + counts_s
            alpha_after_sessions[s] = alpha_work
            belief_after_sessions[s] = alpha_work / alpha_work.sum()
        tau_sessions = solve_tau_vectorized_user(mu_matrix[user_idx], sigma_matrix[user_idx], belief_before_sessions, cost_by_user[user_idx])

        user_static = user_static_by_id[user_id]
        user_rows = []

        for session_idx in range(NUM_SESSIONS):
            ctx = make_context_for_session(session_idx)
            belief_before = belief_before_sessions[session_idx]
            alpha_before = alpha_before_sessions[session_idx]
            belief_after = belief_after_sessions[session_idx]
            alpha_after = alpha_after_sessions[session_idx]
            tau_it = float(tau_sessions[session_idx])
            session_cols = session_cols_matrix[session_idx]
            session_start_time = ctx["time"]
            intersession_gap_h = (
                float((session_start_time - prev_session_end_time).total_seconds() / 3600.0)
                if prev_session_end_time is not None
                else float(continuous_default.get("hist_intersess_gap_h", 0.0))
            )

            for pos0, video_col in enumerate(session_cols):
                video_static = video_static_records[int(video_col)]
                video_id = int(video_ids[video_col])
                level1 = int(level1_col[video_col])
                level2 = int(level2_col[video_col])
                level3 = int(level3_col[video_col])
                cat_idx = int(cat_idx_col[video_col])
                author = video_static.get("i_author_id", "unknown")
                row_time = session_start_time + pd.Timedelta(seconds=8 * pos0)

                has_author_history = int(author in author_last_time)
                hist_last_complete_author = int(author_last_complete.get(author, 0)) if has_author_history else int(binary_default.get("hist_last_complete_author", 0))
                hist_author_recency_days = (
                    float((row_time - author_last_time[author]).total_seconds() / (24.0 * 3600.0))
                    if has_author_history
                    else float(continuous_default.get("hist_author_recency_days", 0.0))
                )
                hist_cat_entropy_l2 = entropy_from_counts(level2_seen_counts.values())

                u_star = float(u_star_matrix[user_idx, video_col])
                p_accept = float(sigmoid_stable((u_star - tau_it) / KAPPA))
                z_star = int(rng.random() < p_accept)

                row = {
                    "user_id": user_id,
                    "video_id": video_id,
                    "user_idx_int": user_idx,
                    "video_idx_int": int(video_col),
                    "session_idx": session_idx,
                    "position_in_session": pos0 + 1,
                    "burst_id": SIM_BURST_ID,
                    "session": session_idx + 1,
                    "sess_rank": float(pos0 + 1),
                    "sess_len": float(VIDEOS_PER_SESSION),
                    "time": row_time,
                    "date": int(row_time.strftime("%Y%m%d")),
                    "timestamp": float(row_time.timestamp()),
                    "level1_category": level1,
                    "level2_category": level2,
                    "level3_category": level3,
                    "i_cat_level1_id": level1,
                    "i_cat_level2_id": level2,
                    "i_cat_level3_id": level3,
                    "cat_idx_int": cat_idx,
                    "c_i": float(cost_by_user[user_idx]),
                    "tau_it": float(tau_it),
                    "mu_hat": float(mu_obs_matrix[user_idx, video_col]),
                    "sigma_hat": float(sigma_obs_matrix[user_idx, video_col]),
                    "u_star": u_star,
                    "p_accept": p_accept,
                    "z_star": z_star,
                    "ctx_hour_sin": ctx["ctx_hour_sin"],
                    "ctx_hour_cos": ctx["ctx_hour_cos"],
                    "ctx_is_weekend": ctx["ctx_is_weekend"],
                    "hist_author_recency_days": hist_author_recency_days,
                    "hist_last_complete_author": hist_last_complete_author,
                    "hist_has_author_history": has_author_history,
                    "hist_prev_sess_len": prev_session_len,
                    "hist_intersess_gap_h": intersession_gap_h,
                    "hist_cat_ema_complete": float(cat_complete_ema.get(level1, continuous_default.get("hist_cat_ema_complete", 0.0))),
                    "hist_cat_entropy_l2": hist_cat_entropy_l2,
                }
                row.update(user_static)
                row.update(video_static)
                row.update(ema_values)
                for k, b in enumerate(belief_before):
                    row[f"belief_before_k_{k}"] = float(b)
                for k, a in enumerate(alpha_before):
                    row[f"alpha_before_k_{k}"] = float(a)

                m0, logv0, m1, logv1 = predict_watch_moments_scalar(row, cat_idx)
                mean = float(m1 if z_star else m0)
                sd = float(np.sqrt(np.exp(logv1 if z_star else logv0)))
                log_wr_star = float(rng.normal(mean, sd))
                log_wr_star = float(np.clip(log_wr_star, LOG_WR_CLIP_LO, LOG_WR_CLIP_HI))
                wr_star = float(np.exp(log_wr_star))
                play_duration = float(wr_star * max(float(row.get("i_video_duration", 0.0) or 0.0), 0.0))
                completion_star = int(wr_star >= COMPLETE_WATCH_RATIO)
                long_star = int((play_duration >= LONG_PLAY_MS) or (wr_star >= LONG_WATCH_RATIO))
                rewatch_star = int(wr_star >= REWATCH_WATCH_RATIO)
                neg_star = int(play_duration <= NEGATIVE_PLAY_MS)

                row.update({
                    "log_wr_star": log_wr_star,
                    "wr_star": wr_star,
                    "watch_ratio": wr_star,
                    "play_duration": play_duration,
                    "completion_star": completion_star,
                    "long_star": long_star,
                    "rewatch_star": rewatch_star,
                    "neg_star": neg_star,
                    "y_complete": completion_star,
                    "y_long": long_star,
                    "y_rewatch": rewatch_star,
                    "y_neg": neg_star,
                })
                user_rows.append(row)

                # Online history update after the current simulated outcome.
                ema_values["hist_ema_y_complete"] = EMA_ALPHA * completion_star + (1.0 - EMA_ALPHA) * ema_values["hist_ema_y_complete"]
                ema_values["hist_ema_y_long"] = EMA_ALPHA * long_star + (1.0 - EMA_ALPHA) * ema_values["hist_ema_y_long"]
                ema_values["hist_ema_y_rewatch"] = EMA_ALPHA * rewatch_star + (1.0 - EMA_ALPHA) * ema_values["hist_ema_y_rewatch"]
                ema_values["hist_ema_y_neg"] = EMA_ALPHA * neg_star + (1.0 - EMA_ALPHA) * ema_values["hist_ema_y_neg"]
                wr_for_history = min(wr_star, HIST_EMA_WATCHRATIO_CAP)
                ema_values["hist_ema_watchratio"] = EMA_ALPHA * wr_for_history + (1.0 - EMA_ALPHA) * ema_values["hist_ema_watchratio"]
                cat_complete_ema[level1] = EMA_ALPHA * completion_star + (1.0 - EMA_ALPHA) * float(cat_complete_ema.get(level1, continuous_default.get("hist_cat_ema_complete", 0.0)))
                level2_seen_counts[level2] = level2_seen_counts.get(level2, 0) + 1
                author_last_complete[author] = completion_star
                author_last_time[author] = row_time

            # Belief was precomputed from the realized category counts and updates after the complete session only.
            prev_session_len = float(VIDEOS_PER_SESSION)
            prev_session_end_time = session_start_time + pd.Timedelta(seconds=8 * (VIDEOS_PER_SESSION - 1))
            for local_idx in range(session_idx * VIDEOS_PER_SESSION, (session_idx + 1) * VIDEOS_PER_SESSION):
                for k, b in enumerate(belief_after):
                    user_rows[local_idx][f"belief_after_k_{k}"] = float(b)
                for k, a in enumerate(alpha_after):
                    user_rows[local_idx][f"alpha_after_k_{k}"] = float(a)

            session_diag_rows.append({
                "user_id": user_id,
                "session_idx": session_idx,
                "tau_it": float(tau_it),
                "mean_belief_entropy_before": entropy_from_counts(belief_before),
            })

        exploration_user_df = pd.DataFrame(user_rows)
        exploration_chunks.append(exploration_user_df)
        category_exploration_counts += np.bincount(exploration_user_df["cat_idx_int"].to_numpy(dtype=np.int64), minlength=len(cat_ids))

        final_belief = belief_after_sessions[-1]
        final_history = {**ema_values}
        final_history.update({
            "hist_cat_entropy_l2": entropy_from_counts(level2_seen_counts.values()),
            "hist_prev_sess_len": float(VIDEOS_PER_SESSION),
            "hist_intersess_gap_h": SIM_SESSION_GAP_HOURS,
        })

        holdout_rows = []
        for video_col in holdout_cols:
            video_static = video_static_records[int(video_col)]
            level1 = int(level1_col[video_col])
            level2 = int(level2_col[video_col])
            level3 = int(level3_col[video_col])
            cat_idx = int(cat_idx_col[video_col])
            author = video_static.get("i_author_id", "unknown")
            has_author_history = int(author in author_last_time)
            row = {
                "user_id": user_id,
                "video_id": int(video_ids[video_col]),
                "user_idx_int": user_idx,
                "video_idx_int": int(video_col),
                "is_holdout": 1,
                "level1_category": level1,
                "level2_category": level2,
                "level3_category": level3,
                "i_cat_level1_id": level1,
                "i_cat_level2_id": level2,
                "i_cat_level3_id": level3,
                "cat_idx_int": cat_idx,
                "burst_id": SIM_BURST_ID,
                "session": NUM_SESSIONS + 1,
                "session_idx": NUM_SESSIONS,
                "position_in_session": 0,
                "sess_rank": 1.0,
                "sess_len": float(VIDEOS_PER_SESSION),
                "ctx_hour_sin": make_context_for_session(NUM_SESSIONS)["ctx_hour_sin"],
                "ctx_hour_cos": make_context_for_session(NUM_SESSIONS)["ctx_hour_cos"],
                "ctx_is_weekend": make_context_for_session(NUM_SESSIONS)["ctx_is_weekend"],
                "c_i": float(cost_by_user[user_idx]),
                "mu_hat": float(mu_obs_matrix[user_idx, video_col]),
                "sigma_hat": float(sigma_obs_matrix[user_idx, video_col]),
                "u_star": float(u_star_matrix[user_idx, video_col]),
                "hist_cat_ema_complete": float(cat_complete_ema.get(level1, continuous_default.get("hist_cat_ema_complete", 0.0))),
                "hist_author_recency_days": float(continuous_default.get("hist_author_recency_days", 0.0)),
                "hist_last_complete_author": int(author_last_complete.get(author, 0)) if has_author_history else int(binary_default.get("hist_last_complete_author", 0)),
                "hist_has_author_history": has_author_history,
            }
            row.update(user_static)
            row.update(video_static)
            row.update(final_history)
            for k, b in enumerate(final_belief):
                row[f"belief_final_k_{k}"] = float(b)
            holdout_rows.append(row)
        holdout_user_df = pd.DataFrame(holdout_rows)
        holdout_chunks.append(holdout_user_df)
        category_holdout_counts += np.bincount(holdout_user_df["cat_idx_int"].to_numpy(dtype=np.int64), minlength=len(cat_ids))

        all_diag_rows.append({
            "user_id": user_id,
            "exploration_rows": int(len(exploration_user_df)),
            "holdout_rows": int(len(holdout_user_df)),
            "mean_wr_star": float(exploration_user_df["wr_star"].mean()),
            "mean_completion_star": float(exploration_user_df["completion_star"].mean()),
            "mean_u_star_exploration": float(exploration_user_df["u_star"].mean()),
            "mean_u_star_holdout": float(holdout_user_df["u_star"].mean()),
        })

    exploration_chunk_df = pd.concat(exploration_chunks, ignore_index=True)
    holdout_chunk_df = pd.concat(holdout_chunks, ignore_index=True)

    append_parquet(exploration_chunk_df, EXPLORATION_OUTPUT_PATH, exploration_writer)
    append_parquet(holdout_chunk_df, HOLDOUT_OUTPUT_PATH, holdout_writer)
    print(f"wrote users [{user_start}, {user_stop}) exploration={len(exploration_chunk_df):,} holdout={len(holdout_chunk_df):,}")

close_writer(exploration_writer)
close_writer(holdout_writer)
print("saved exploration:", EXPLORATION_OUTPUT_PATH)
print("saved holdout:", HOLDOUT_OUTPUT_PATH)

wrote users [0, 50) exploration=100,000 holdout=50,000


wrote users [50, 100) exploration=100,000 holdout=50,000


wrote users [100, 150) exploration=100,000 holdout=50,000


wrote users [150, 200) exploration=100,000 holdout=50,000


wrote users [200, 250) exploration=100,000 holdout=50,000


...

wrote users [1550, 1600) exploration=100,000 holdout=50,000


wrote users [1600, 1650) exploration=100,000 holdout=50,000


wrote users [1650, 1700) exploration=100,000 holdout=50,000


wrote users [1700, 1750) exploration=100,000 holdout=50,000


wrote users [1750, 1777) exploration=54,000 holdout=27,000
saved exploration: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_exploration_interactions.parquet
saved holdout: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_holdout_candidates.parquet


## 5. Save Manifest, Config, and Diagnostics

This section writes the validation artifacts and checks that the simulated environment is internally consistent.

The manifest classifies feature columns into continuous, binary, categorical, history, and belief groups. The recommender notebooks use this manifest instead of rediscovering feature definitions from the data files.

The diagnostics report row counts, candidate counts per user, utility distributions, simulated watch-ratio distributions, label rates, and feature availability. These checks are useful for confirming that the validation set is balanced before training any recommender.


In [6]:
exploration_probe = pd.read_parquet(EXPLORATION_OUTPUT_PATH)
holdout_probe = pd.read_parquet(HOLDOUT_OUTPUT_PATH)

belief_before_cols = [c for c in exploration_probe.columns if c.startswith("belief_before_k_")]
belief_after_cols = [c for c in exploration_probe.columns if c.startswith("belief_after_k_")]
belief_final_cols = [c for c in holdout_probe.columns if c.startswith("belief_final_k_")]
alpha_cols = [c for c in exploration_probe.columns if c.startswith("alpha_before_k_") or c.startswith("alpha_after_k_")]

id_columns = ["user_id", "video_id", "user_idx_int", "video_idx_int", "session_idx", "position_in_session"]
measurement_only_columns = [
    "sess_rank", "sess_len", "position_in_session", "log1p_i_video_duration",
    "sess_rank_over_sess_len", "sess_rank_over_sess_len_sq",
]
label_columns = [
    "z_star", "p_accept", "log_wr_star", "wr_star",
    "completion_star", "long_star", "rewatch_star", "neg_star",
    "watch_ratio", "play_duration", "y_complete", "y_long", "y_rewatch", "y_neg",
]
ground_truth_columns = ["u_star", "mu_hat", "sigma_hat", "tau_it", "c_i"]
history_feature_columns = [c for c in mlp_feature_columns if c.startswith("hist_")]
static_user_feature_columns = [c for c in mlp_feature_columns if c.startswith("u_")]
static_video_feature_columns = [c for c in mlp_feature_columns if c.startswith("i_")]
belief_feature_columns = belief_before_cols + belief_after_cols + belief_final_cols + alpha_cols

manifest = {
    "mlp_feature_columns": mlp_feature_columns,
    "continuous_cols": continuous_cols,
    "binary_cols": binary_cols,
    "categorical_cols": categorical_cols,
    "static_user_feature_columns": static_user_feature_columns,
    "static_video_feature_columns": static_video_feature_columns,
    "history_feature_columns": history_feature_columns,
    "belief_feature_columns": belief_feature_columns,
    "measurement_only_columns": measurement_only_columns,
    "label_columns": label_columns,
    "ground_truth_columns": ground_truth_columns,
    "id_columns": id_columns,
    "do_not_use_for_ranking_unless_position_aware": ["position_in_session", "sess_rank"],
    "feature_groups": {
        **{c: "static_user_feature" for c in static_user_feature_columns},
        **{c: "static_video_feature" for c in static_video_feature_columns},
        **{c: "history_feature" for c in history_feature_columns},
        **{c: "belief_feature" for c in belief_feature_columns},
        **{c: "measurement_only_feature" for c in measurement_only_columns},
        **{c: "label_or_ground_truth" for c in label_columns + ground_truth_columns},
    },
}
MANIFEST_OUTPUT_PATH.write_text(json.dumps(manifest, indent=2))

config = {
    "num_users": int(N_users),
    "num_videos": int(N_videos),
    "num_exploration_videos_per_user": NUM_EXPLORATION_VIDEOS_PER_USER,
    "num_holdout_videos_per_user": NUM_HOLDOUT_VIDEOS_PER_USER,
    "videos_per_session": VIDEOS_PER_SESSION,
    "num_sessions": NUM_SESSIONS,
    "prior_strength": PRIOR_STRENGTH,
    "kappa": KAPPA,
    "random_seed": RANDOM_SEED,
    "ema_alpha": EMA_ALPHA,
    "log_wr_clip_lo": LOG_WR_CLIP_LO,
    "log_wr_clip_hi": LOG_WR_CLIP_HI,
    "hist_ema_watchratio_cap": HIST_EMA_WATCHRATIO_CAP,
    "input_paths": {
        "processed_data": str(PROCESSED_DATA_PATH),
        "gnn_data": str(GNN_DATA_PATH),
        "true_utility": str(TRUE_UTILITY_PATH),
        "utility_params": str(UTILITY_PARAM_PATH),
        "structural_params": str(STRUCTURAL_PARAM_PATH),
    },
    "output_paths": {
        "exploration": str(EXPLORATION_OUTPUT_PATH),
        "holdout": str(HOLDOUT_OUTPUT_PATH),
        "config": str(CONFIG_OUTPUT_PATH),
        "manifest": str(MANIFEST_OUTPUT_PATH),
        "diagnostics": str(DIAGNOSTICS_OUTPUT_PATH),
    },
}
CONFIG_OUTPUT_PATH.write_text(json.dumps(config, indent=2))

exploration = exploration_probe
holdout = holdout_probe

category_distribution_exploration = pd.DataFrame({
    "cat_idx_int": np.arange(len(cat_ids)),
    "level1_category": cat_ids,
    "n_exploration": category_exploration_counts,
    "share_exploration": category_exploration_counts / category_exploration_counts.sum(),
    "n_holdout": category_holdout_counts,
    "share_holdout": category_holdout_counts / category_holdout_counts.sum(),
})

session_moments = (
    exploration
    .groupby("session_idx", observed=True)
    .agg(
        n=("user_id", "size"),
        wr_star_mean=("wr_star", "mean"),
        completion_rate=("completion_star", "mean"),
        p_accept_mean=("p_accept", "mean"),
        z_star_mean=("z_star", "mean"),
        tau_mean=("tau_it", "mean"),
        u_star_mean=("u_star", "mean"),
    )
    .reset_index()
)

diagnostics = {
    "exploration_rows": int(len(exploration)),
    "holdout_rows": int(len(holdout)),
    "num_users_exploration": int(exploration["user_id"].nunique()),
    "num_users_holdout": int(holdout["user_id"].nunique()),
    "unique_videos_exploration": int(exploration["video_id"].nunique()),
    "unique_videos_holdout": int(holdout["video_id"].nunique()),
    "u_star_exploration_mean": float(exploration["u_star"].mean()),
    "u_star_exploration_std": float(exploration["u_star"].std()),
    "u_star_holdout_mean": float(holdout["u_star"].mean()),
    "u_star_holdout_std": float(holdout["u_star"].std()),
    "tau_it_mean": float(exploration["tau_it"].mean()),
    "tau_it_std": float(exploration["tau_it"].std()),
    "p_accept_mean": float(exploration["p_accept"].mean()),
    "z_star_mean": float(exploration["z_star"].mean()),
    "wr_star_mean": float(exploration["wr_star"].mean()),
    "wr_star_std": float(exploration["wr_star"].std()),
    "completion_rate": float(exploration["completion_star"].mean()),
    "long_rate": float(exploration["long_star"].mean()),
    "rewatch_rate": float(exploration["rewatch_star"].mean()),
    "neg_rate": float(exploration["neg_star"].mean()),
    "category_distribution": category_distribution_exploration.to_dict(orient="records"),
    "session_moments_head": session_moments.head(20).to_dict(orient="records"),
    "session_moments_tail": session_moments.tail(20).to_dict(orient="records"),
    "user_moments_head": pd.DataFrame(all_diag_rows).head(20).to_dict(orient="records"),
}
DIAGNOSTICS_OUTPUT_PATH.write_text(json.dumps(diagnostics, indent=2))

print("saved config:", CONFIG_OUTPUT_PATH)
print("saved manifest:", MANIFEST_OUTPUT_PATH)
print("saved diagnostics:", DIAGNOSTICS_OUTPUT_PATH)
print(json.dumps({k: v for k, v in diagnostics.items() if k not in ["category_distribution", "session_moments_head", "session_moments_tail", "user_moments_head"]}, indent=2))
display(category_distribution_exploration.head())
display(session_moments.head())

saved config: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_simulation_config.json
saved manifest: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_feature_manifest.json
saved diagnostics: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/validation_simulation/validation_simulation_diagnostics.json
{
  "exploration_rows": 3554000,
  "holdout_rows": 1777000,
  "num_users_exploration": 1777,
  "num_users_holdout": 1777,
  "unique_videos_exploration": 10728,
  "unique_videos_holdout": 10728,
  "u_star_exploration_mean": 1.9524309260173316,
  "u_star_exploration_std": 1.9401401307775197,
  "u_star_holdout_mean": 1.9523263618656723,
  "u_star_holdout_std": 1.940388546362385,
  "tau_it_mean": 1.6971185586950184,
  "tau_it_std": 1.0424066242116723,
  "p_accept_mean": 0.5968335873701464,
  "z_star_mean": 0.5967121553179516,
  "wr_star_mean": 1.0197482259323705,
  "wr_star_std"

,cat_idx_int,level1_category,n_exploration,share_exploration,n_holdout,share_holdout
0,0,-124,14286,0.004020,7025,0.003953
1,1,1,139646,0.039293,70096,0.039446
2,2,2,63160,0.017772,31538,0.017748
3,3,3,34654,0.009751,17322,0.009748
4,4,4,50893,0.014320,25561,0.014384


,session_idx,n,wr_star_mean,completion_rate,p_accept_mean,z_star_mean,tau_mean,u_star_mean
0,0,17770,1.103439,0.375746,0.591314,0.589252,1.697828,1.944150
1,1,17770,1.070771,0.356837,0.596643,0.597243,1.693294,1.957670
2,2,17770,1.054808,0.355768,0.601671,0.601745,1.695022,1.975006
3,3,17770,1.016546,0.335228,0.593190,0.593191,1.699144,1.935826
4,4,17770,1.025811,0.341868,0.597009,0.598143,1.697711,1.956173
